<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">04 Mobile Algorithms</h1>
<hr>


## **3. PID for Rocket Landing**

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>


### 3.1 Introduction

Getting a rocket to the ground is easy. Getting it there slowly and on the landing pad is the challenge, especially when gusts push it sideways. Too little thrust leads to a hard landing. Too much thrust can send the rocket climbing again. Steering toward the pad also tilts the thrust away from the vertical, leaving less of it available to slow the descent.

In [04.2 PID Controller](04.2_PID_Controller.ipynb), you explored how proportional, integral, and derivative feedback affect a system. Here you will tune two PID controllers to land a small simulated rocket. One controls the descent speed; the other steers toward the pad. Your challenge is to **find the highest Horizontal Wind setting at which your rocket still lands successfully most of the time**.

Start with a gentle landing in calm air, then turn up the wind and improve your gains. Aim for roughly **four successful landings out of five attempts** with different seeds. This is a practical tuning challenge, not a precise reliability measurement. Try things, watch what happens, use a plot when you get stuck, and see how far you can push your controller.


### 3.2 Watch a Real Landing

On December 21, 2015, SpaceX landed a Falcon 9 first stage back on land after launching satellites into orbit. Watch the final descent in SpaceX's official recap, [“The Falcon has landed”](https://www.youtube.com/watch?v=ANv5UfZsvZQ). Notice how the rocket must arrive at the right position **and** reduce its speed before contact.

### 3.3 Your Mission

Your rocket starts **100 m above a landing pad at $x=0$**, initially at rest. Height $y$ is positive above the ground; vertical velocity $v_y$ is negative during descent. Your task is to find the strongest sideways gusts your controller can handle while still landing gently most of the time.

**The wind challenge:** Work toward the highest **Horizontal Wind** setting where the same six PID gains succeed in about **4 out of 5 new attempts**. Keep the gains and wind setting unchanged during those five attempts, and let the seed increment each time. Count crashes and timeouts as failed attempts. Increase the wind, retune, and try again; if success becomes rare, step back to your last reliable setting. A rough tally is enough. You do not need to prove an absolute maximum or calculate statistical confidence.

The simulation uses a point mass in two dimensions. The mass stays at 1 kg, gravity is constant, and the engine can produce 0 to 20 N of thrust. Position and velocity are measured perfectly. Thrust and its direction change instantly; fuel use, air resistance, rocket rotation, landing legs, and communication delays are omitted. These choices let us concentrate on feedback control.

| Part of the landing | Target or rule in this notebook |
| :--- | :--- |
| Descent-speed controller | Follow $v_y^* = -0.2\,\mathrm{m/s} - (0.1\,\mathrm{s}^{-1})\,y$: descend faster high up and slow down near the ground. |
| Steering controller | Reduce the horizontal position error $e_x = 0-x$ by tilting the thrust. |
| Horizontal Wind | Strength of random sideways gusts, expressed as the standard deviation of horizontal acceleration in m/s². A larger setting means stronger gusts; it is not a wind speed. |
| Successful touchdown | Reach $y=0$ within 2 m of the pad center, with both $\lvert v_y\rvert\leq0.5$ m/s and $\lvert v_x\rvert\leq0.5$ m/s. |
| End of a run | Stop at first ground contact, or report a timeout after 500 s. |

The descent rule is a **setpoint**, not the actual velocity. The rocket only follows it if the controller produces suitable thrust. At 100 m the target is -10.2 m/s; near the ground it approaches -0.2 m/s. The small downward target allows the rocket to reach the ground instead of hovering just above it.

The vertical PID output commands thrust in newtons. The horizontal PID output is converted to a thrust angle with `arctan2(output, 1)`; positive angles push to the right.

Horizontal wind also affects vertical speed because the rocket **tilts to counter the sideways push**. Its thrust then points partly sideways, leaving less thrust pushing upward:

$$F_{\text{up}} = T\cos(\theta)$$

Here, $T$ is the total engine thrust and $\theta$ is the tilt angle measured from vertical. At a **45° tilt, only about 71% of the thrust acts upward**. During descent, the rocket therefore falls faster unless the vertical PID increases thrust to compensate.

That compensation takes time, and the engine has a **20 N limit**. Strong steering can therefore make a gentle landing harder. This is how the two PID controllers interact: a change that improves the horizontal position can make the vertical descent worse.


### 3.4 Run and Tune the Simulation

Choose **Run All** to load the cockpit, or run the collapsed launch cell under **Your Interactive Landing Lab**. Use the six gain sliders to tune the descent-speed and steering controllers. Releasing a slider updates the simulation, replay, and diagnostic plots together. You can also click a slider's number to enter a precise value.

The **Horizontal Wind** slider starts at 1.0 m/s². Set it to 0.0 m/s² to learn the controls, then increase it and work toward your highest reliable wind setting. Change one gain at a time and select **Keep seed fixed** while comparing settings. The starting gains are deliberately untuned.

Click **Play new attempt** for another landing. **Increment seed each play** is selected by default: each new played attempt advances the displayed seed, including attempts started automatically when you change a gain or Horizontal Wind. Select **Keep seed fixed** for reproducible wind with the same settings. Entering a seed directly uses that exact value. **Replay**, **Play**, and the timeline revisit the recorded flight without changing its seed.

The replay shows the rocket, its thrust flame, and the path it has travelled. The Swiss flag beside the flight path points downwind, flutters more strongly in gusts, and droops in calm conditions. It follows a smoothed version of the recorded wind so that rapid gust changes remain readable; the wind-push reading is in m/s², matching the model. The close-up focuses on the landing pad. Use **Play/Pause**, **Restart**, the timeline, and the playback-speed selector to examine the final approach. Longer flights initially play faster; the clock always shows simulated seconds. Turn off **Play after changes** if you prefer to start each replay yourself. Open **Flight diagnostics** below the replay to inspect the four plots.

**Sound effects:** Sound is on by default. If the browser requires an initial interaction, the replay waits at **Play**. Click it once; later gain-slider updates reuse the same audio player and keep your sound and volume settings. Engine rumble follows the thrust. A failed touchdown triggers a cartoon explosion with an impact sound. A successful landing plays a happy chime followed by a short recording of real applause. Use the volume slider or click **Sound: on** to mute.

The simulation uses NumPy, Matplotlib, and ipywidgets from the course environment. The replay is drawn locally and works without an internet connection. A live Python kernel is needed to change gains; rerun the launch cell after reopening the notebook to reconnect the controls.

Thrust is limited to 20 N. This basic PID implementation does not prevent **integral windup**: accumulated error can keep growing while the requested thrust exceeds that limit. If the rocket reacts too late or climbs again, compare the requested thrust with the applied thrust in the plots.


#### Your Interactive Landing Lab

The two columns contain $K_p$, $K_i$, and $K_d$ for the descent-speed and steering controllers. The gains belong to different loops and do not have interchangeable units. Keep the model constants fixed for the challenge.

First get a landing at **Horizontal Wind = 0.0 m/s²**. Then try 0.5 m/s² and build upward in steps of about 0.5 m/s², using smaller steps near your limit. Select **Keep seed fixed** when diagnosing a failure so you can see what a gain change does. Switch back to **Increment seed each play** and use **Play new attempt** for a quick five-attempt check with your chosen gains. Replaying the same flight does not count as another attempt.

The aim is a high wind setting with roughly four successes out of five, not one lucky landing. Record your best wind setting, gains, seeds, and a simple success tally. If you reach the top of the slider, that is the strongest wind available in this challenge.

Run the collapsed cell below to open the cockpit. All implementation code lives in the [helper](../../.admin/notebooks/rocket_landing/lab.py); you can complete the challenge using the controls without editing code.


In [ ]:
from pathlib import Path

_helper = next(root / ".admin/notebooks/rocket_landing/lab.py"
               for root in (Path.cwd(), *Path.cwd().parents)
               if (root / ".admin/notebooks/rocket_landing/lab.py").is_file())
get_ipython().run_line_magic("run", f'-i "{_helper.as_posix()}"')


Open **Flight diagnostics** and read the panels together. A trajectory that reaches the ground does not necessarily mean a successful landing: check the pad offset and both velocity components at contact. The green velocity band matters at touchdown, not throughout the descent. Large steering angles reduce the upward component of thrust, and an engine request above the limit cannot be applied.

The plotted trajectory uses different scales on its horizontal and vertical axes so that small sideways movements remain visible. It is a diagnostic plot, not a drawing of the rocket to scale.

The replay interpolates between recorded states for smooth motion. Its enlarged rocket symbol shows the commanded thrust direction, not a simulated body orientation. Landing success still uses the same position and velocity checks as the plots.


### 3.5 References

- [04.2 PID Controller](04.2_PID_Controller.ipynb) - the preceding control exercise and PID equation.
- [SpaceX: “The Falcon has landed”](https://www.youtube.com/watch?v=ANv5UfZsvZQ) - official launch and landing recap.
- [SpaceX: Falcon 9](https://www.spacex.com/vehicles/falcon-9/) - the real launch vehicle that inspired this exercise.



----
### 3.6 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


**Set Horizontal Wind to 0.0 m/s² and try the starting gains. What goes wrong? Use the animation or Flight diagnostics to identify whether the rocket descends too fast, climbs again, or misses the pad. Which controller would you change first?**

(answer here)


**Find gains that produce a gentle landing in calm air. Change one gain at a time and record a three trials. What did you notice when changing the proportional, integral, or derivative action?**

| Trial | Descent-speed $(K_p, K_i, K_d)$ | Outcome | What changed? |
| :--- | :--- | :--- | :--- |
| 1 | | | |
| 2 | | | |
| 3 | | | |

(answer here)


**Raise Horizontal Wind to 0.5 m/s² and select Keep seed fixed. Tune the steering controller and, if needed, the descent-speed controller until you can land. Record a useful gain change. Why can stronger steering make the vertical landing worse?**

(answer here)


**OPTIONAL: How much horizontal wind can your rocket handle? Take on the wind challenge. Increase Horizontal Wind, retune your gains, and find the highest setting where you land successfully in roughly 4 out of 5 new attempts. For each five-attempt check, select Increment seed each play, keep all six gains and the wind setting unchanged, and click Play new attempt five times. Record the seeds actually used and your success tally. What is your best result?**

Hint: Keep the model constants and touchdown rules fixed; tune only the six PID gains and Horizontal Wind. Use a fixed seed to debug and changing seeds to check your final settings. Use roughly 0.5 m/s² steps at first and smaller steps near your limit. Count every attempt in a check, including crashes and timeouts. A few notes and a rough tally are enough: these runs give a practical impression of consistency, and a formal percentage calculation is not required. The goal is to build a feel for PID control and have fun experimenting.

| Horizontal Wind (m/s²) | Descent-speed $(K_p, K_i, K_d)$ | Steering $(K_p, K_i, K_d)$ | Seeds used | Successful landings / 5 |
| :--- | :--- | :--- | :--- | :--- |
| | | | | |
| | | | | |
| | | | | |

(answer here)


**OPTIONAL: Try a little more wind than your best setting. What starts to fail first: pad position, sideways speed, or descent speed? Use one failed flight to explain a gain change you would try next. If you already reached the top of the slider, inspect a difficult attempt there instead.**

Hint: Keep the seed fixed so you can compare the same gust sequence before and after a gain change. Inspect the position and velocity at touchdown in Flight diagnostics. Sideways oscillation suggests a different steering adjustment from a steady drift; a fast descent during strong tilting may mean too little thrust is directed upward. Change one gain and replay the same seed to check your explanation.

(answer here)


**OPTIONAL: Bonus challenge: find a windy attempt where the requested thrust reaches or exceeds the engine limit. How could the integral term keep accumulating and affect the later flight? Suggest a tuning change that might help.**

Hint: Compare requested thrust with the engine limit in Flight diagnostics. While the engine is already at maximum thrust, a continuing velocity error can still build up the integral term. Consider what happens when that error becomes small or changes sign. Try reducing the descent-speed integral gain while keeping the other gains and seed fixed, then compare the flight.

(answer here)


**OPTIONAL: Did you read through and understand all the tasks in this notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

(please help the team by telling us what part was not understood, and we will work on it)
